[← Back to 09 - Thinking in Metrics](<../09-thinking-in-metrics.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 09 - Thinking in Metrics: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/09-thinking-in-metrics-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 1. Two kinds of average order value

In [3]:
%%sql
SELECT AVG(amount) AS avg_order_value
FROM orders;

avg_order_value
82.0


In [4]:
%%sql
WITH customer_averages AS (
    SELECT
        customer_id,
        AVG(amount) AS customer_avg
    FROM orders
    GROUP BY
        customer_id
)
SELECT ROUND(AVG(customer_avg), 2) AS avg_of_customer_averages
FROM customer_averages;

avg_of_customer_averages
91.67


82.0 and 91.67. The first is 410 across five orders. The second is the average of Ada's 50, Bola's 85, and Chidi's 140. They differ because the second gives every customer an equal vote, whatever their number of orders: Chidi's single order of 140 counts for one fifth of the first average but one third of the second, and it's the largest order in the table, so it pulls the second number up. Neither query is broken. The first describes a typical order, and the second describes a typical ordering customer's typical order, and a dashboard tile labelled "average order value" should be the first one unless someone has decided otherwise and written that down. (Section 9 shows the gap going the other way on AdventureWorks.)

## 2. RANK vs DENSE_RANK

In [5]:
%%sql
WITH customer_orders AS (
    SELECT
        c.customer_name,
        COUNT(o.order_id) AS order_count
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    order_count,
    RANK() OVER (ORDER BY order_count DESC) AS order_rank,
    DENSE_RANK() OVER (ORDER BY order_count DESC) AS order_dense_rank
FROM customer_orders
ORDER BY
    order_count DESC,
    customer_name;

customer_name,order_count,order_rank,order_dense_rank
Ada,2,1,1
Bola,2,1,1
Chidi,1,3,2
Dami,0,4,3


Ada and Bola share first place in both. Chidi is 3rd by `RANK()`, because two customers are ahead of him, and 2nd by `DENSE_RANK()`, which never leaves gaps. Use `RANK()` when the position should say how many are ahead ("third-best customer"), and `DENSE_RANK()` when you want tiers ("the second-highest order count"). `ROW_NUMBER()` would give Ada and Bola different numbers, which is exactly the silent tiebreaker section 6 warned about.

## 3. Share of all revenue

In [6]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent,
    ROUND(100.0 * total_spent / SUM(total_spent) OVER (), 1) AS pct_of_revenue
FROM customer_totals
ORDER BY
    customer_id;

customer_name,total_spent,pct_of_revenue
Ada,100,24.4
Bola,170,41.5
Chidi,140,34.1
Dami,0,0.0


24.4, 41.5, 34.1, and 0.0, adding to 100. Three earlier ideas meet here: `COALESCE` keeps Dami as a real 0 rather than a `NULL`, `SUM() OVER ()` from `06` puts the grand total on every row, and `100.0` stops the division from truncating every share to 0.

## 4. Stretch: the best customer of 2013 (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [7]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [8]:
%%tsql
WITH totals_2013 AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS total_2013
    FROM Sales.SalesOrderHeader
    WHERE OrderDate >= '2013-01-01'
      AND OrderDate < '2014-01-01'
    GROUP BY
        CustomerID
),
totals_all_time AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS total_all_time
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
)
SELECT TOP 3
    t.CustomerID,
    t.total_2013,
    a.total_all_time,
    RANK() OVER (ORDER BY t.total_2013 DESC) AS rank_2013,
    (SELECT COUNT(*) FROM totals_all_time AS x WHERE x.total_all_time > a.total_all_time) + 1 AS rank_all_time
FROM totals_2013 AS t
INNER JOIN totals_all_time AS a
    ON a.CustomerID = t.CustomerID
ORDER BY
    t.total_2013 DESC;

CustomerID,total_2013,total_all_time,rank_2013,rank_all_time
29641,475912.5591,606819.6087,1,27
29913,470230.9927,757449.6804,2,15
29818,431068.4796,989184.0820,3,1


Customer 29641 tops 2013, while the all-time leader, 29818, is only third in 2013. Same data, same measure, different period, different answer. The date filter uses `>= '2013-01-01'` and `< '2014-01-01'` rather than `BETWEEN`, which is the safe habit with dates that might carry a time of day, since `BETWEEN '2013-01-01' AND '2013-12-31'` would miss anything after midnight on 31 December.

---

[← Back to 09 - Thinking in Metrics](<../09-thinking-in-metrics.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).